In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D 

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

def construct_model(
  arch,
  train_embed_layer,
  act,
  width,
  hid_n_layers,
):
  # arch : "MLP", "ResNet", "DiffNet-Add", "DiffNet-Mul"
  # train_embed_layer : bool, whether to train the first layer
  # act : "tanh", "sin", "relu", "gelu"
  # width : int, number of hidden units in each layer
  # hid_n_layers : int, number of hidden layers
  if arch == "MLP":
    return MLP(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      residual=False
    )
  elif arch == "ResNet":
    return MLP(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      residual=True
    )
  elif arch == "DiffNet-Add":
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='add'
    )
  elif arch == "DiffNet-Mul":
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='mul'
    )
  elif arch.startswith("DiffNet-Const"):
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='const',
      scale_const=float(arch.split('-')[-1]) if '-' in arch else 1.0
    )

class MLP(nn.Module):
  
  def __init__(self,
    act='relu',
    train_embed_layer=False,
    width=64,
    hid_n_layers=2,
    residual=False
  ):
    super(MLP, self).__init__()
    self.train_embed_layer = train_embed_layer
    self.width = width
    self.hid_n_layers = hid_n_layers
    self.residual = residual

    if act.lower() == 'relu':
      self.act = nn.ReLU()
    elif act.lower() == 'gelu':
      self.act = nn.GELU()
    elif act.lower() == 'tanh':
      self.act = nn.Tanh()
    elif act.lower() == 'sin':
      self.act = torch.sin

    self.embed_layer = nn.Linear(2, width, bias=True)
    if not self.train_embed_layer:
      self.embed_layer.weight.requires_grad = False
      self.embed_layer.bias.requires_grad = False
    
    self.hidden_layers = nn.ModuleList()
    for _ in range(hid_n_layers - 1):
      self.hidden_layers.append(nn.Linear(width, width, bias=True))
    
    self.output_layer = nn.Linear(width, 1, bias=True)
  
  def forward(self, scalar_input1, scalar_input2):
    scalar_input1 = scalar_input1.unsqueeze(-1)
    scalar_input2 = scalar_input2.unsqueeze(-1)

    combined_input = torch.cat((scalar_input1, scalar_input2), dim=-1)

    x = self.embed_layer(combined_input)
    x = torch.sin(x) 
    for layer in self.hidden_layers:
      h = layer(x)
      if self.residual:
        x = self.act(h) + x
      else:
        x = self.act(h)
    x = self.output_layer(x)
    return x  
class DiffNet(nn.Module):
  
  def __init__(self,
    act='relu',
    train_embed_layer=False,
    width=64,
    hid_n_layers=2,
    scale_type='add',
    scale_const=1.0,
  ):
    super(DiffNet, self).__init__()

    self.begin_embed_layer = nn.Linear(2, width, bias=True)
    if not train_embed_layer:
      self.begin_embed_layer.weight.requires_grad = False
      self.begin_embed_layer.bias.requires_grad = False
    
    self.embed_layers = nn.ModuleList()
    self.hidden_layers = nn.ModuleList()
    for i in range(hid_n_layers):
      embed_layer = nn.Linear(2, width, bias=True)
      if scale_type == 'add':
        embed_layer.weight.data *= (i + 2)
        embed_layer.bias.data *= (i + 2)
      elif scale_type == 'mul':
        embed_layer.weight.data *= 2**(i + 1)
        embed_layer.bias.data *= 2**(i + 1)
      elif scale_type == 'const' and scale_const != 1.0:
        embed_layer.weight.data *= scale_const
        embed_layer.bias.data *= scale_const
      
      if not train_embed_layer:
        embed_layer.weight.requires_grad = False
        embed_layer.bias.requires_grad = False
      self.embed_layers.append(embed_layer)

      self.hidden_layers.append(nn.Linear(2 * width, width, bias=True))
    
    self.output_layer = nn.Linear(width, 1, bias=True)

    if act.lower() == 'relu':
      self.act = nn.ReLU()
    elif act.lower() == 'gelu':
      self.act = nn.GELU()
    elif act.lower() == 'tanh':
      self.act = nn.Tanh()
    elif act.lower() == 'sin':
      self.act = torch.sin
  
  def forward(self, scalar_input1, scalar_input2):
    scalar_input1 = scalar_input1.unsqueeze(-1)
    scalar_input2 = scalar_input2.unsqueeze(-1)

    combined_input = torch.cat((scalar_input1, scalar_input2), dim=-1)

    x = self.begin_embed_layer(combined_input)
    x = torch.sin(x)
    for embed_layer, hidden_layer in zip(self.embed_layers, self.hidden_layers):
      h = embed_layer(combined_input)
      h = torch.sin(h)

      z = torch.cat((x, h), dim=-1)
      z = hidden_layer(z)
      x = self.act(z) + x

    x = self.output_layer(x)
    return x + 5


In [ ]:
first_coordinates = [0, 1, 2, 3, 4, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
second_coordinates = [1, 2, 3, 4, 0, 5, 6, 7, 8, 9, 7, 8, 9, 5, 6]

first_bp_coordinates = [0, 1, 2, 3, 4, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9] + [10, 11, 12, 13, 14, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
second_bp_coordinates = [11, 12, 13, 14, 10, 15, 16, 17, 18, 19, 17, 18, 19, 15, 16] + [1, 2, 3, 4, 0, 5, 6, 7, 8, 9, 7, 8, 9, 5, 6]

first_heawood_coordinates = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 0, 1, 2, 3, 4, 5, 7]
second_heawood_coordinates = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 0, 9, 6, 11, 8, 13, 10, 12]

from itertools import permutations

pair_to_ind = {}
for pair in torch.combinations(torch.arange(14), r=2):
  current_size = len(pair_to_ind) // 2 
  pair_to_ind[(pair[0].item(), pair[1].item())] = current_size
  pair_to_ind[(pair[1].item(), pair[0].item())] = current_size

heawood_perm_indices = [] 

for mul in [1, 3, 5]:
  l = []
  for i in range(len(first_heawood_coordinates)):
    u = (first_heawood_coordinates[i] * mul) % 14
    v = (second_heawood_coordinates[i] * mul) % 14
    l.append(pair_to_ind[(u, v)])
  heawood_perm_indices.append(l)

  l = [] 
  for i in range(len(first_heawood_coordinates)):
    u = (first_heawood_coordinates[i] * mul + 1) % 14
    v = (second_heawood_coordinates[i] * mul + 1) % 14
    l.append(pair_to_ind[(u, v)])

bp_perm_indices = [] 

petersen_perm_indices = [
  [0, 28, 9, 3, 24, 21, 34, 40, 41, 36, 17, 43, 4, 13, 37],
  [0, 28, 21, 12, 40, 11, 5, 17, 43, 1, 33, 38, 41, 32, 25],
  [6, 28, 9, 2, 31, 15, 43, 32, 1, 20, 35, 33, 38, 16, 25],
  [18, 27, 15, 2, 31, 17, 23, 14, 39, 35, 4, 33, 38, 7, 16],
  [18, 24, 37, 31, 8, 5, 23, 14, 39, 4, 42, 10, 16, 25, 22],
  [10, 6, 18, 3, 24, 12, 5, 14, 44, 29, 35, 19, 41, 42, 22],
  [0, 6, 12, 34, 39, 30, 44, 20, 29, 26, 7, 19, 10, 32, 22],
  [34, 27, 15, 8, 11, 36, 1, 30, 20, 29, 26, 13, 7, 19, 42],
  [9, 3, 21, 37, 27, 40, 2, 8, 11, 36, 23, 30, 44, 26, 13]
]

ind_to_pair = {}
ind = 0
for pair in torch.combinations(torch.arange(10), r=2):
  ind_to_pair[ind] = (pair[0].item(), pair[1].item())
  ind += 1

pair_to_ind = {}
for pair in torch.combinations(torch.arange(20), r=2):
  current_size = len(pair_to_ind) // 2 
  pair_to_ind[(pair[0].item(), pair[1].item())] = current_size
  pair_to_ind[(pair[1].item(), pair[0].item())] = current_size

for perm in petersen_perm_indices:
  l = []
  for ind in perm:
    u, v = ind_to_pair[ind]
    l.append(pair_to_ind[(u, v + 10)])
    l.append(pair_to_ind[(v, u + 10)])
  bp_perm_indices.append(l)

def subset_product_mean(tensor, subset_size):
  coeffs = torch.zeros(subset_size + 1, device=tensor.device)
  coeffs[0] = 1.0

  size_vec = torch.arange(1, subset_size + 1, device=tensor.device)

  for i in range(len(tensor)):
    coeffs[1:] = (((i+1) - size_vec) / (i+1)) * coeffs[1:] + (size_vec / (i + 1)) * coeffs[:-1] * tensor[i]

  return coeffs[subset_size]

def log_subset_product_mean(log_tensor, subset_size, verbose=False):
  N = log_tensor.shape[0]
  k = subset_size

  log_coeffs = torch.full((k + 1,), -float('inf'), device=log_tensor.device, dtype=log_tensor.dtype)
  log_coeffs[0] = 0.0

  size_vec = torch.arange(1, k + 1, device=log_tensor.device, dtype=log_tensor.dtype)

  iterable = range(N)
  
  for i in iterable:
    n = i + 1
    log_x_n = log_tensor[i]

    max_j = min(n, k)
    
    current_size_vec = size_vec[:max_j]
    
    log_c1 = torch.log((n - current_size_vec) / n)
    term1 = log_c1 + log_coeffs[1 : max_j + 1]

    log_c2 = torch.log(current_size_vec / n)
    term2 = log_c2 + log_coeffs[0 : max_j] + log_x_n

    stacked_terms = torch.stack([term1, term2], dim=0)
    log_coeffs[1 : max_j + 1] = torch.logsumexp(stacked_terms, dim=0)

  return log_coeffs[k]

def compute_heawood_density(model, exponent, num_samples, device):
  
  points = torch.rand([num_samples, 14], device=device)

  indices = torch.combinations(torch.arange(14, device=device), r=2)

  pairs_first = points[:, indices[:, 0]] 
  pairs_second = points[:, indices[:, 1]] 

  new_x = torch.minimum(pairs_first, pairs_second)
  new_y = torch.maximum(pairs_first, pairs_second)

  pair_densities = model(new_x, new_y).squeeze(2) # [num_samples, num_edges, 1]

  pair_densities = torch.nn.functional.logsigmoid(pair_densities)

  edge_densities = pair_densities[:, heawood_perm_indices] # [num_samples, 3, 21]

  graph_densities = torch.logsumexp(torch.sum(edge_densities, dim=2), dim=1) - np.log(edge_densities.shape[1])

  graph_power_densities = log_subset_product_mean(graph_densities, exponent) # []

  return torch.exp(graph_power_densities)

def compute_bipartised_petersen_density(model, exponent, num_samples, device):
  
  points = torch.rand([num_samples, 20], device=device)

  indices = torch.combinations(torch.arange(20, device=device), r=2)

  pairs_first = points[:, indices[:, 0]] 
  pairs_second = points[:, indices[:, 1]] 

  new_x = torch.minimum(pairs_first, pairs_second)
  new_y = torch.maximum(pairs_first, pairs_second)

  pair_densities = model(new_x, new_y).squeeze(-2) 

  pair_densities = torch.nn.functional.logsigmoid(pair_densities)

  edge_densities = pair_densities[:, bp_perm_indices] 

  graph_densities = torch.logsumexp(torch.sum(edge_densities, dim=2), dim=1) - np.log(edge_densities.shape[1])

  graph_power_densities = log_subset_product_mean(graph_densities, exponent) 

  return torch.exp(graph_power_densities)

In [ ]:
heawood_exponent = 10
bp_exponent = 7
from scipy import special

def test_ineq(model, num_batch, batch_size):
  
  with torch.no_grad():
    heawood_densities = compute_heawood_density(model, num_batch, batch_size, heawood_exponent, DEVICE).flatten()
    bp_densities = compute_bipartised_petersen_density(model, num_batch, batch_size, bp_exponent, DEVICE).flatten()
  print()
  mean_heawood_densities = np.mean(np.exp(heawood_densities))
  logmean_heawood_densities = special.logsumexp(heawood_densities) - np.log(len(heawood_densities))
  print(f"Mean Heawood Density : {mean_heawood_densities}")
  print(f"LogMean Heawood Density : {np.exp(logmean_heawood_densities)}")
  print(f"Powered Mean Heawood Density : {mean_heawood_densities ** (heawood_exponent / bp_exponent)}")
  print(f"Powered LogMean Heawood Density : {np.exp(logmean_heawood_densities * (heawood_exponent / bp_exponent))}")
  print()

  mean_bp_densities = np.mean(np.exp(bp_densities))
  logmean_bp_densities = special.logsumexp(bp_densities) - np.log(len(bp_densities))
  print(f"Mean BP Density : {mean_bp_densities}")
  print(f"LogMean BP Density : {np.exp(logmean_bp_densities)}")
  print(f"Powered Mean BP Density : {mean_bp_densities ** (bp_exponent / heawood_exponent)}")
  print(f"Powered LogMean BP Density : {np.exp(logmean_bp_densities * (bp_exponent / heawood_exponent))}")
  
  print()

  print("Results : (If negative, we disprove the conjecture)")

  print(f"Heawood - PoweredBP : {mean_heawood_densities - (mean_bp_densities ** (bp_exponent / heawood_exponent))}")
  print(f"LogHeawood - PoweredLogBP : {np.exp(logmean_heawood_densities) - np.exp(logmean_bp_densities * (bp_exponent / heawood_exponent))}")

  print(f"PoweredHeawood - BP : {(mean_heawood_densities ** (heawood_exponent / bp_exponent)) - mean_bp_densities}")
  print(f"PoweredLogHeawood - LogBP : {np.exp(logmean_heawood_densities * (heawood_exponent / bp_exponent)) - np.exp(logmean_bp_densities)}")
  print()

In [ ]:
import os

TEST_MC_SAMPLES = 2 ** 14

for file in os.listdir():
  if file.endswith(".pt"):
    print(file)
    
    file_split = file[:-3].split("_")

    arch = file_split[4]
    act = file_split[5]
    width = int(file_split[6])
    hid_n_layers = int(file_split[7])
    lr = float(file_split[8])

    model = construct_model(
      arch=arch,
      train_embed_layer=False,
      act=act,
      width=width,
      hid_n_layers=hid_n_layers
    ).to(DEVICE)

    model.load_state_dict(torch.load(file, map_location=DEVICE))
    model.eval()

    test_ineq(model, 2 ** 14, TEST_MC_SAMPLES)